# Token Skip & Early Exit Evaluation

在 GSM8K 上评测 Token Skip 和 Early Exit 优化。

**Token Skip 原理：**
- 基于 dual_cache，比较当前 hidden state 与上一 step 的相似度
- 使用最后四层的 cos_sim 判定（硬编码阈值 0.75）
- 满足条件的 token 复用上一 step 的 hidden state
- 上一步跳过的 token，这一步必须重算（防止误差累积）

**Early Exit 原理：**
- 基于 dual_cache，在 `early_exit_layer` 层后判定
- 比较前 `early_exit_layer` 层的平均 cos_sim 与 `early_exit_threshold`
- 满足条件的 token 后续层复用上一 step 结果
- 上一步跳过的 token，这一步必须重算（防止误差累积）

**实验配置:**
- Token Skip: 硬编码阈值 0.75，测试不同 `force_full_every_k`
- Early Exit: 测试不同的 `early_exit_threshold`

In [ ]:
import os
import torch
import gc

# Set GPU (modify as needed)
os.environ['CUDA_VISIBLE_DEVICES'] = '0,1,2,3,4,5,6,7'

# Environment settings
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
os.environ['HF_ALLOW_CODE_EVAL'] = '1'
os.environ['HF_DATASETS_TRUST_REMOTE_CODE'] = 'true'

# Change to llada directory
os.chdir('llada')

# Create log directory
os.makedirs('nlogs', exist_ok=True)

# Clear GPU cache
torch.cuda.empty_cache()
gc.collect()

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

## 1. Token Skip 测试（快速测试 limit=30）

In [ ]:
import subprocess
import datetime

task = "gsm8k"
fewshot = 3
limit = 30
gpu = 0

# Token Skip 测试（阈值硬编码为 0.75）
name = "token_skip_test"
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
log_file = f"nlogs/{task}_{name}_{timestamp}.log"

model_args = [
    "model_path='GSAI-ML/LLaDA-8B-Instruct'",
    "gen_length=128",
    "steps=32",
    "block_length=32",
    "threshold=0.9",
    "use_cache=True",
    "dual_cache=True",
    "token_skip=True",
    "force_full_every_k=3",
    "show_speed=True",
]

cmd = f"""CUDA_VISIBLE_DEVICES={gpu} accelerate launch eval_llada.py \\
    --tasks {task} --num_fewshot {fewshot} --limit {limit} \\
    --confirm_run_unsafe_code --model llada_dist \\
    --model_args {','.join(model_args)}"""

print(f"\n{'='*60}")
print(f"Running: {name}")
print(f"Log file: {log_file}")
print('='*60)

with open(log_file, 'w') as f:
    result = subprocess.run(cmd, shell=True, stdout=f, stderr=subprocess.STDOUT, text=True)

with open(log_file, 'r') as f:
    lines = f.readlines()
    print("Last 15 lines:")
    for line in lines[-15:]:
        print(line.rstrip())

## 2. Early Exit 快速测试（limit=30）

In [ ]:
import subprocess
import datetime

task = "gsm8k"
fewshot = 3
limit = 30
gpu = 1

# Early Exit 测试
threshold = 0.95
name = f"early_exit_th{int(threshold*100)}"
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
log_file = f"nlogs/{task}_{name}_{timestamp}.log"

model_args = [
    "model_path='GSAI-ML/LLaDA-8B-Instruct'",
    "gen_length=128",
    "steps=32",
    "block_length=32",
    "threshold=0.9",
    "use_cache=True",
    "dual_cache=True",
    "early_exit=True",
    "early_exit_layer=16",
    f"early_exit_threshold={threshold}",
    "force_full_every_k=4",
    "show_speed=True",
]

cmd = f"""CUDA_VISIBLE_DEVICES={gpu} accelerate launch eval_llada.py \\
    --tasks {task} --num_fewshot {fewshot} --limit {limit} \\
    --confirm_run_unsafe_code --model llada_dist \\
    --model_args {','.join(model_args)}"""

print(f"\n{'='*60}")
print(f"Running: {name}")
print(f"Log file: {log_file}")
print('='*60)

with open(log_file, 'w') as f:
    result = subprocess.run(cmd, shell=True, stdout=f, stderr=subprocess.STDOUT, text=True)

with open(log_file, 'r') as f:
    lines = f.readlines()
    print("Last 15 lines:")
    for line in lines[-15:]:
        print(line.rstrip())

In [ ]:
torch.cuda.empty_cache()

## 3. GSM8K 并行测试 - Early Exit（7个 threshold）

In [ ]:
import subprocess
import datetime

task = "gsm8k"
fewshot = 3
limit = 500

# (early_exit_threshold, gpu, name)
# 7个不同的 early_exit_threshold 值
configs = [
    (1.00, 1, "early_exit_th100"),   # baseline (no skip)
    (0.99, 2, "early_exit_th99"),
    (0.98, 3, "early_exit_th98"),
    (0.97, 4, "early_exit_th97"),
    (0.96, 5, "early_exit_th96"),
    (0.95, 6, "early_exit_th95"),
    (0.94, 7, "early_exit_th94"),
]

processes = []

for threshold, gpu, name in configs:
    log_file = f"nlogs/early_exit_{task}_{name}_{datetime.datetime.now():%F_%H-%M-%S}.log"
    
    model_args = [
        "model_path='GSAI-ML/LLaDA-8B-Instruct'",
        "gen_length=128",
        "steps=32",
        "block_length=32",
        "threshold=0.9",
        "use_cache=True",
        "dual_cache=True",
        "early_exit=True",
        "early_exit_layer=16",
        f"early_exit_threshold={threshold}",
        "force_full_every_k=4",
        "show_speed=True",
    ]
    
    cmd = f"""CUDA_VISIBLE_DEVICES={gpu} accelerate launch eval_llada.py \\
        --tasks {task} --num_fewshot {fewshot} --limit {limit}\\
        --confirm_run_unsafe_code --model llada_dist \\
        --model_args {','.join(model_args)} \\
        --output_path evals_results/early_exit/gsm8k-{name} --log_samples"""
    
    print(f"GPU{gpu} running: {name} (threshold={threshold})")
    p = subprocess.Popen(cmd, shell=True, stdout=open(log_file, "w"), stderr=subprocess.STDOUT)
    processes.append((p, name, log_file))

print(f"\n{len(processes)} tasks launched, waiting...")

for p, name, log in processes:
    p.wait()
    print(f"{name} finished (exit code: {p.returncode})")

## 4. GSM8K 并行测试 - Token Skip（测试不同 force_full_every_k）

In [ ]:
import subprocess
import datetime

task = "gsm8k"
fewshot = 3
limit = 500

# (force_full_every_k, gpu, name)
# 测试不同的 force_full_every_k 值
configs = [
    (0, 1, "token_skip_k0"),    # 禁用强制全算
    (2, 2, "token_skip_k2"),
    (3, 3, "token_skip_k3"),
    (4, 4, "token_skip_k4"),
    (5, 5, "token_skip_k5"),
    (6, 6, "token_skip_k6"),
]

processes = []

for force_k, gpu, name in configs:
    log_file = f"nlogs/token_skip_{task}_{name}_{datetime.datetime.now():%F_%H-%M-%S}.log"
    
    model_args = [
        "model_path='GSAI-ML/LLaDA-8B-Instruct'",
        "gen_length=128",
        "steps=32",
        "block_length=32",
        "threshold=0.9",
        "use_cache=True",
        "dual_cache=True",
        "token_skip=True",
        f"force_full_every_k={force_k}",
        "show_speed=True",
    ]
    
    cmd = f"""CUDA_VISIBLE_DEVICES={gpu} accelerate launch eval_llada.py \\
        --tasks {task} --num_fewshot {fewshot} --limit {limit}\\
        --confirm_run_unsafe_code --model llada_dist \\
        --model_args {','.join(model_args)} \\
        --output_path evals_results/token_skip/gsm8k-{name} --log_samples"""
    
    print(f"GPU{gpu} running: {name} (force_full_every_k={force_k})")
    p = subprocess.Popen(cmd, shell=True, stdout=open(log_file, "w"), stderr=subprocess.STDOUT)
    processes.append((p, name, log_file))

print(f"\n{len(processes)} tasks launched, waiting...")

for p, name, log in processes:
    p.wait()
    print(f"{name} finished (exit code: {p.returncode})")

## 5. GSM8K 结果查看 - Early Exit

In [ ]:
import glob
import re
import os

log_files = sorted(glob.glob("nlogs/early_exit_gsm8k*.log"), key=os.path.getmtime, reverse=True)

print("Early Exit GSM8K Results:")
print("=" * 110)

results = []
for log_file in log_files:
    name = os.path.basename(log_file)
    
    with open(log_file, 'r') as f:
        content = f.read()
    
    # Accuracy - 匹配表格格式: |flexible-extract|...|0.566|±|
    acc_match = re.search(r'flexible-extract.*?\|(\d+\.\d+)\|±', content)
    acc = float(acc_match.group(1)) * 100 if acc_match else None
    
    # Speed
    speed_match = re.search(r'Tokens per second:\s*([\d.]+)', content)
    speed = float(speed_match.group(1)) if speed_match else None
    
    # NFE
    nfe_match = re.search(r'Total NFE is (\d+)', content)
    nfe = int(nfe_match.group(1)) if nfe_match else None
    
    # Time
    time_match = re.search(r'Total time taken:\s*([\d.]+)', content)
    time_sec = float(time_match.group(1)) if time_match else None
    
    # 从文件名提取 threshold
    th_match = re.search(r'th(\d+)', name)
    if th_match:
        th_str = th_match.group(1)
        threshold = int(th_str) / 100  # th95 -> 0.95
    else:
        threshold = None
    
    results.append({
        'name': name,
        'threshold': threshold,
        'accuracy': acc,
        'tokens_per_sec': speed,
        'total_nfe': nfe,
        'time_sec': time_sec,
    })

# 按 threshold 排序
results.sort(key=lambda x: x['threshold'] if x['threshold'] else 0, reverse=True)

print(f"{'Name':<55} {'Thresh':<8} {'Acc%':<8} {'Tok/s':<10} {'NFE':<10} {'Time(s)':<10}")
print("-" * 110)
for r in results:
    th_str = f"{r['threshold']:.2f}" if r['threshold'] else "N/A"
    acc_str = f"{r['accuracy']:.2f}" if r['accuracy'] else "N/A"
    speed_str = f"{r['tokens_per_sec']:.1f}" if r['tokens_per_sec'] else "N/A"
    nfe_str = str(r['total_nfe']) if r['total_nfe'] else "N/A"
    time_str = f"{r['time_sec']:.1f}" if r['time_sec'] else "N/A"
    print(f"{r['name']:<55} {th_str:<8} {acc_str:<8} {speed_str:<10} {nfe_str:<10} {time_str:<10}")

## 6. GSM8K 结果查看 - Token Skip

In [ ]:
import glob
import re
import os

log_files = sorted(glob.glob("nlogs/token_skip_gsm8k*.log"), key=os.path.getmtime, reverse=True)

print("Token Skip GSM8K Results:")
print("=" * 110)

results = []
for log_file in log_files:
    name = os.path.basename(log_file)
    
    with open(log_file, 'r') as f:
        content = f.read()
    
    # Accuracy
    acc_match = re.search(r'flexible-extract.*?\|(\d+\.\d+)\|±', content)
    acc = float(acc_match.group(1)) * 100 if acc_match else None
    
    # Speed
    speed_match = re.search(r'Tokens per second:\s*([\d.]+)', content)
    speed = float(speed_match.group(1)) if speed_match else None
    
    # NFE
    nfe_match = re.search(r'Total NFE is (\d+)', content)
    nfe = int(nfe_match.group(1)) if nfe_match else None
    
    # Time
    time_match = re.search(r'Total time taken:\s*([\d.]+)', content)
    time_sec = float(time_match.group(1)) if time_match else None
    
    # 从文件名提取 force_full_every_k
    k_match = re.search(r'_k(\d+)', name)
    force_k = int(k_match.group(1)) if k_match else None
    
    results.append({
        'name': name,
        'force_k': force_k,
        'accuracy': acc,
        'tokens_per_sec': speed,
        'total_nfe': nfe,
        'time_sec': time_sec,
    })

# 按 force_k 排序
results.sort(key=lambda x: x['force_k'] if x['force_k'] is not None else 999)

print(f"{'Name':<55} {'K':<6} {'Acc%':<8} {'Tok/s':<10} {'NFE':<10} {'Time(s)':<10}")
print("-" * 110)
for r in results:
    k_str = str(r['force_k']) if r['force_k'] is not None else "N/A"
    acc_str = f"{r['accuracy']:.2f}" if r['accuracy'] else "N/A"
    speed_str = f"{r['tokens_per_sec']:.1f}" if r['tokens_per_sec'] else "N/A"
    nfe_str = str(r['total_nfe']) if r['total_nfe'] else "N/A"
    time_str = f"{r['time_sec']:.1f}" if r['time_sec'] else "N/A"
    print(f"{r['name']:<55} {k_str:<6} {acc_str:<8} {speed_str:<10} {nfe_str:<10} {time_str:<10}")

## 7. MBPP 测试 - GPU 池 + Threshold 池

### 7.1 Early Exit on MBPP

<!-- 如需测试 MBPP，将 task 改为 "mbpp"，其他参数基本相同 -->

In [ ]:
import subprocess
import datetime
import threading
import queue
import time

# ============== 配置区 ==============
task = "mbpp"  # 改为 mbpp
fewshot = 3
limit = 400  # None = 全量测试

# GPU 池子 - 可用的 GPU 列表
gpu_pool = [1, 2, 3, 4, 5, 6, 7]

# Early Exit Threshold 池子
threshold_pool = [
    1.00,  # baseline
    0.99, 0.98, 0.97, 0.96, 0.95, 0.94, 0.93, 0.92, 0.91, 0.90,
]

# ============== 任务队列 ==============
task_queue = queue.Queue()
for th in threshold_pool:
    task_queue.put(th)

results_lock = threading.Lock()
results = []

def get_threshold_name(th):
    """根据 threshold 生成文件名后缀，如 0.95 -> th95"""
    return f"th{int(th * 100)}"

def worker(gpu_id):
    """每个 GPU 的工作线程"""
    while True:
        try:
            threshold = task_queue.get_nowait()
        except queue.Empty:
            break
        
        name = f"early_exit_{get_threshold_name(threshold)}"
        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        log_file = f"nlogs/early_exit_{task}_{name}_{timestamp}.log"
        
        model_args = [
            "model_path='GSAI-ML/LLaDA-8B-Instruct'",
            "gen_length=128",
            "steps=32",
            "block_length=32",
            "threshold=0.9",
            "use_cache=True",
            "dual_cache=True",
            "early_exit=True",
            "early_exit_layer=16",
            f"early_exit_threshold={threshold}",
            "force_full_every_k=4",
            "show_speed=True",
        ]
        
        limit_arg = f"--limit {limit}" if limit else ""
        
        cmd = f"""CUDA_VISIBLE_DEVICES={gpu_id} accelerate launch eval_llada.py \\
            --tasks {task} --num_fewshot {fewshot} {limit_arg} \\
            --confirm_run_unsafe_code --model llada_dist \\
            --model_args {','.join(model_args)} \\
            --output_path evals_results/early_exit/{task}-{name} --log_samples"""
        
        print(f"[GPU {gpu_id}] 开始: threshold={threshold} -> {log_file}")
        start_time = time.time()
        
        with open(log_file, 'w') as f:
            result = subprocess.run(cmd, shell=True, stdout=f, stderr=subprocess.STDOUT)
        
        elapsed = time.time() - start_time
        print(f"[GPU {gpu_id}] 完成: threshold={threshold}, 耗时={elapsed/60:.1f}分钟, exit_code={result.returncode}")
        
        with results_lock:
            results.append({
                'gpu': gpu_id,
                'threshold': threshold,
                'log_file': log_file,
                'exit_code': result.returncode,
                'elapsed_min': elapsed / 60
            })
        
        task_queue.task_done()

# ============== 启动所有 GPU worker ==============
print(f"任务总数: {len(threshold_pool)}")
print(f"可用 GPU: {gpu_pool}")
print(f"Task: {task}, fewshot: {fewshot}, limit: {limit}")
print("=" * 60)

threads = []
for gpu_id in gpu_pool:
    t = threading.Thread(target=worker, args=(gpu_id,))
    t.start()
    threads.append(t)
    time.sleep(2)  # 错开启动，避免同时加载模型

# 等待所有任务完成
for t in threads:
    t.join()

print("\n" + "=" * 60)
print("所有任务完成！")
print(f"成功: {sum(1 for r in results if r['exit_code'] == 0)}/{len(results)}")

# 按 threshold 排序显示结果
results.sort(key=lambda x: x['threshold'], reverse=True)
print(f"\n{'Threshold':<12} {'GPU':<6} {'耗时(分)':<12} {'Exit':<6} {'Log File'}")
print("-" * 80)
for r in results:
    print(f"{r['threshold']:<12.4f} {r['gpu']:<6} {r['elapsed_min']:<12.1f} {r['exit_code']:<6} {r['log_file']}")

### 7.2 Token Skip on MBPP

<!-- 如需测试 Token Skip 在 MBPP 上的效果，复制上面的代码块并将：
1. 将 early_exit=True 改为 token_skip=True
2. 移除 early_exit_layer 和 early_exit_threshold
3. threshold_pool 改为测试不同的 force_full_every_k 值
-->

## 8. MBPP 结果查看

In [ ]:
import glob
import re
import os

log_files = sorted(glob.glob("nlogs/early_exit_mbpp*.log"), key=os.path.getmtime, reverse=True)

print("Early Exit MBPP Results:")
print("=" * 120)

mbpp_results = []
for log_file in log_files[:50]:
    name = os.path.basename(log_file)
    
    with open(log_file, 'r') as f:
        content = f.read()
    
    # MBPP 使用 pass@1 指标
    acc_match = re.search(r'pass@1.*?\|(\d+\.\d+)\|±', content)
    if not acc_match:
        acc_match = re.search(r'\|mbpp\|.*?\|(\d+\.\d+)\|±', content)
    acc = float(acc_match.group(1)) * 100 if acc_match else None
    
    # Speed
    speed_match = re.search(r'Tokens per second:\s*([\d.]+)', content)
    speed = float(speed_match.group(1)) if speed_match else None
    
    # NFE
    nfe_match = re.search(r'Total NFE is (\d+)', content)
    nfe = int(nfe_match.group(1)) if nfe_match else None
    
    # Time
    time_match = re.search(r'Total time taken:\s*([\d.]+)', content)
    time_sec = float(time_match.group(1)) if time_match else None
    
    # 从文件名提取 threshold
    th_match = re.search(r'th(\d+)', name)
    if th_match:
        threshold = int(th_match.group(1)) / 100
    else:
        threshold = None
    
    mbpp_results.append({
        'name': name,
        'threshold': threshold,
        'accuracy': acc,
        'tokens_per_sec': speed,
        'total_nfe': nfe,
        'time_sec': time_sec,
    })

# 按 threshold 排序
mbpp_results.sort(key=lambda x: x['threshold'] if x['threshold'] else 0, reverse=True)

print(f"{'Name':<55} {'Thresh':<8} {'Acc%':<8} {'Tok/s':<10} {'NFE':<10} {'Time(s)':<10}")
print("-" * 120)
for r in mbpp_results:
    th_str = f"{r['threshold']:.2f}" if r['threshold'] else "N/A"
    acc_str = f"{r['accuracy']:.2f}" if r['accuracy'] else "N/A"
    speed_str = f"{r['tokens_per_sec']:.1f}" if r['tokens_per_sec'] else "N/A"
    nfe_str = str(r['total_nfe']) if r['total_nfe'] else "N/A"
    time_str = f"{r['time_sec']:.1f}" if r['time_sec'] else "N/A"
    print(f"{r['name']:<55} {th_str:<8} {acc_str:<8} {speed_str:<10} {nfe_str:<10} {time_str:<10}")

## 9. 结果可视化

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 需要先运行上面的结果查看 cell 获取 results 数据

# 从 Early Exit GSM8K 结果中提取有效数据
log_files = sorted(glob.glob("nlogs/early_exit_gsm8k*.log"), key=os.path.getmtime, reverse=True)

plot_data = []
for log_file in log_files:
    name = os.path.basename(log_file)
    
    with open(log_file, 'r') as f:
        content = f.read()
    
    acc_match = re.search(r'flexible-extract.*?\|(\d+\.\d+)\|±', content)
    acc = float(acc_match.group(1)) * 100 if acc_match else None
    
    speed_match = re.search(r'Tokens per second:\s*([\d.]+)', content)
    speed = float(speed_match.group(1)) if speed_match else None
    
    th_match = re.search(r'th(\d+)', name)
    threshold = int(th_match.group(1)) / 100 if th_match else None
    
    if acc is not None and threshold is not None:
        plot_data.append({
            'threshold': threshold,
            'accuracy': acc,
            'speed': speed,
        })

# 按 threshold 排序
plot_data.sort(key=lambda x: x['threshold'], reverse=True)

if plot_data:
    thresholds = [d['threshold'] for d in plot_data]
    accuracies = [d['accuracy'] for d in plot_data]
    speeds = [d['speed'] for d in plot_data if d['speed']]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Accuracy vs Threshold
    ax1.plot(thresholds, accuracies, 'bo-', linewidth=2, markersize=10)
    ax1.set_xlabel('Early Exit Threshold', fontsize=12)
    ax1.set_ylabel('Accuracy (%)', fontsize=12)
    ax1.set_title('GSM8K Accuracy vs Early Exit Threshold', fontsize=14)
    ax1.grid(True, alpha=0.3)
    ax1.invert_xaxis()  # 从高到低

    # Speed vs Threshold
    if speeds:
        ax2.plot(thresholds[:len(speeds)], speeds, 'ro-', linewidth=2, markersize=10)
        ax2.set_xlabel('Early Exit Threshold', fontsize=12)
        ax2.set_ylabel('Tokens/sec', fontsize=12)
        ax2.set_title('GSM8K Speed vs Early Exit Threshold', fontsize=14)
        ax2.grid(True, alpha=0.3)
        ax2.invert_xaxis()

    plt.tight_layout()
    plt.savefig('nlogs/early_exit_results.png', dpi=150)
    plt.show()
    
    print("\n图表已保存至 nlogs/early_exit_results.png")
    
    # 打印数据表格
    print("\n" + "="*60)
    print(f"{'Threshold':<12} {'Accuracy %':<15} {'Speed':<15}")
    print("-"*60)
    for d in plot_data:
        speed_str = f"{d['speed']:.1f}" if d['speed'] else "N/A"
        print(f"{d['threshold']:<12.2f} {d['accuracy']:<15.2f} {speed_str:<15}")
else:
    print("没有找到有效的 Early Exit 结果数据")